In [ ]:
!pip -q install transformers pandas

GPU and Drive

In [2]:
import os, json, numpy as np, pandas as pd, torch
assert torch.cuda.is_available(), "GPU missing: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))
from google.colab import drive; drive.mount("/content/drive")
D = "/content/drive/MyDrive/spoilfree/"
print(os.listdir(D))

Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['val.csv', 'eval_handwritten_107.csv', 'train.csv', 'test._heldout.csv']


Settings

In [6]:
teacher="bhavyagiri/roberta-base-finetuned-imdb-spoilers"
student="roberta-base"
maxlen=160
keep_pos=0.35 #keep top 35% according to teacher
keep_neg=0.70 #keep 70% of the least spoiler content
epochs=3
bs=32
lr=2e-5
seed=1

try:
    test_file=D+"test_heldout.csv"
    if not os.path.exists(test_file):
        test_file="/content/test._heldout.csv"
    if not os.path.exists(test_file):
        raise FileNotFoundError("test._heldout.csv not found")
    print("Using:", test_file)
except FileNotFoundError as e:
    print(f"File not found: {e}")

torch.manual_seed(seed)
np.random.seed(seed)

Using: /content/test._heldout.csv


**Helper Functions**

In [3]:
from sys import thread_info
from sklearn.metrics import roc_auc_score, average_precision_score

def clean_split(d,keep_posnew=None,keep_negnew=None,seed=1):
  keep_posnew=keep_pos if keep_posnew is None else keep_posnew
  keep_negnew=keep_neg if keep_negnew is None else keep_negnew
  pos,neg=d[d.label==1],d[d.label==0]
  pos=pos[pos.t1>=pos.t1.quantile(1-keep_posnew)]
  neg=neg[neg.t1>=neg.t1.quantile(keep_negnew)]
  k=min(len(pos),len(neg))
  out=pd.concat(pos.sample(k,random_state=seed),neg.sample(k,random_state=seed))
  return out.sample(frac=1,random_state=seed).reset_index(drop=True)[["text","label","t1"]]


def sweep(y,p,name,ths=(0.3,0.5,0.7,0.8,0.9,0.95)):
  y=np.asarray(y)
  p=np.asarray(p)
  print(f"{name} AUC: {roc_auc_score(y,p):.3f} AP {average_precision_score(y,p):.3f}")
  print(f"thr precision recall falseblur missed")
  for ts in ths:
    pr=p>=ts
    tp=int((pr & (y==1)).sum())
    fp=int((pr & (y==0)).sum())       #confusion matrix metric
    fn=int(((~pr) & (y==1)).sum())
    tn=int(((~pr) & (y==0)).sum())
    print(f"{ts:<6} {tp / max(tp + fp, 1):<10.3f} {tp / max(tp + fn, 1):<7.3f} {fp:<10} {fn}")


def pick_threshold(y,p,min_precision=0.95):
  #lowest precision that is >=min_precision if no threshold reaches then None
  y=np.asarray(y)
  p=np.asarray(p)
  for ts in np.arange(0.05,1.0,0.01):
    pr=p>=ts
    if pr.sum() and (pr&(y==1)).sum()/pr.sum()>=min_precision:
      return round(float(ts),2)
  return None




**Score from the teacher**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
